# Notebook 2: Experimental Tests
## Comparing Mean and Osculating Orbital Elements

We observe the time evolution of the mean orbital elements provided in a Two-Line Element (TLE) and the osculating elements, which are the SGP4-propagated mean elements with secular evolution and short- as well as long-period periodic variational corrections appended. This helps better visualise what the mean orbital elements capture, compared to the osculating elements captured by the instantaneous satellite states that account for perturbations.

In this notebook, I will focus on comparing the mean and osculating orbital elements specifically for LAGEOS-1, a Medium Earth Orbit (MEO) satellite, where the TLEs were sourced from [Space-Track](www.space-track.org) for the year 2025. Hence, the annual evolution for the year 2025 will be investigated.

### Extracting Mean Orbital Elements for LAGEOS-1

I begin by first extracting the mean orbital elements for LAGEOS-1 for every TLE scraped between January 1st and December 31st 2025. The TLEs are first sorted by epoch, removing duplicates and retaining the TLE with the latest creation date for more precise ephemeris.

In [ ]:
# General Imports:
import os
import sys
import subprocess
import numpy as np
import pandas as pd
from pathlib import Path

# Establish Java path to remove error:
os.environ["JAVA_HOME"] = str(Path(sys.prefix) / "Library" / "jre")

# Repo paths + frame:
data_path = Path.cwd().resolve().parent / "data"
download_path = Path.cwd().resolve().parent / "download"

In [2]:
# TLE paths for LAGEOS-1:
tle_path = download_path / "tle" / "8820.json"
df = pd.read_json(tle_path)

df.columns

Index(['CCSDS_OMM_VERS', 'COMMENT', 'CREATION_DATE', 'ORIGINATOR',
       'OBJECT_NAME', 'OBJECT_ID', 'CENTER_NAME', 'REF_FRAME', 'TIME_SYSTEM',
       'MEAN_ELEMENT_THEORY', 'EPOCH', 'MEAN_MOTION', 'ECCENTRICITY',
       'INCLINATION', 'RA_OF_ASC_NODE', 'ARG_OF_PERICENTER', 'MEAN_ANOMALY',
       'EPHEMERIS_TYPE', 'CLASSIFICATION_TYPE', 'NORAD_CAT_ID',
       'ELEMENT_SET_NO', 'REV_AT_EPOCH', 'BSTAR', 'MEAN_MOTION_DOT',
       'MEAN_MOTION_DDOT', 'SEMIMAJOR_AXIS', 'PERIOD', 'APOAPSIS', 'PERIAPSIS',
       'OBJECT_TYPE', 'RCS_SIZE', 'COUNTRY_CODE', 'LAUNCH_DATE', 'SITE',
       'DECAY_DATE', 'FILE', 'GP_ID', 'TLE_LINE0', 'TLE_LINE1', 'TLE_LINE2'],
      dtype='str')

In [3]:
# Sort TLEs + remove duplicate epochs:
df["EPOCH"] = pd.to_datetime(df["EPOCH"], utc=True).dt.tz_localize(None)
df["CREATION_DATE"] = pd.to_datetime(df["CREATION_DATE"], utc=True)
df = df[(df["EPOCH"] >= "2025-01-01") & (df["EPOCH"] < "2026-01-01")]  # only 2025
df = df.sort_values(["EPOCH", "CREATION_DATE"])
df = df.drop_duplicates("EPOCH", keep="last")  # drop stalest TLEs at duplicate epochs

# Extract mean elements from the TLEs:
e = df["ECCENTRICITY"].to_numpy(dtype=float)
i = df["INCLINATION"].to_numpy(dtype=float)
raan = df["RA_OF_ASC_NODE"].to_numpy(dtype=float)
omega = df["ARG_OF_PERICENTER"].to_numpy(dtype=float)
M = np.deg2rad(df["MEAN_ANOMALY"].to_numpy(dtype=float))

df.head(5)

,CCSDS_OMM_VERS,COMMENT,CREATION_DATE,ORIGINATOR,OBJECT_NAME,OBJECT_ID,CENTER_NAME,REF_FRAME,TIME_SYSTEM,MEAN_ELEMENT_THEORY,...,RCS_SIZE,COUNTRY_CODE,LAUNCH_DATE,SITE,DECAY_DATE,FILE,GP_ID,TLE_LINE0,TLE_LINE1,TLE_LINE2
0,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-01-01 23:41:24+00:00,18 SPCS,LAGEOS 1,1976-039A,EARTH,TEME,UTC,SGP4,...,MEDIUM,US,1976-05-04,AFWTR,NaN,4598149,277043280,0 LAGEOS 1,1 08820U 76039A 25001.13974220 .00000019 0...,2 08820 109.8523 357.0926 0044138 55.8219 315...
1,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-01-02 19:02:11+00:00,18 SPCS,LAGEOS 1,1976-039A,EARTH,TEME,UTC,SGP4,...,MEDIUM,US,1976-05-04,AFWTR,NaN,4598496,277110144,0 LAGEOS 1,1 08820U 76039A 25002.45726355 .00000019 0...,2 08820 109.8531 357.5433 0044100 55.5246 105...
3,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-01-07 18:07:06+00:00,18 SPCS,LAGEOS 1,1976-039A,EARTH,TEME,UTC,SGP4,...,MEDIUM,US,1976-05-04,AFWTR,NaN,4603301,277521054,0 LAGEOS 1,1 08820U 76039A 25007.23315350 .00000025 0...,2 08820 109.8536 359.1775 0044183 54.4700 285...
4,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-01-08 20:49:28+00:00,18 SPCS,LAGEOS 1,1976-039A,EARTH,TEME,UTC,SGP4,...,MEDIUM,US,1976-05-04,AFWTR,NaN,4604166,277598650,0 LAGEOS 1,1 08820U 76039A 25008.55898025 .00000023 0...,2 08820 109.8539 359.6312 0044151 54.1770 94...
5,3.0,GENERATED VIA SPACE-TRACK.ORG API,2025-01-09 20:58:34+00:00,18 SPCS,LAGEOS 1,1976-039A,EARTH,TEME,UTC,SGP4,...,MEDIUM,US,1976-05-04,AFWTR,NaN,4604746,277699556,0 LAGEOS 1,1 8820U 76039A 25009.26847366 .00000021 0...,2 8820 109.8542 359.8740 0044129 54.0129 285...


Having extracted $e, i, \Omega, \omega$, and $M$, will now be initialising the semi-major axis term $a$ that is independent of time from the Kozai to Brouwer convention. This includes initialising key constants, such as $J_2$. These constants as well as equations implemented in the cell below are all reproduced from Appendix B of Hoots et al. (2004).

In [4]:
# Recover SGP4 mean semimajor axis from Hoots et al. (2004):
n0 = df["MEAN_MOTION"].to_numpy(dtype=float)*2*np.pi/1440
ke = 0.0743669161  # Earth radii^(3/2)/min
J2 = 1.082616e-3
Re = 6378.135 # Earth radius [km]

a1 = (ke/n0)**(2/3)
beta = np.sqrt(1 - e**2)
delta1 = 0.75*J2*(3*np.cos(np.deg2rad(i))**2 - 1)/(a1**2 * beta**3)
a2 = a1*(1 - delta1/3 - delta1**2 - 134*delta1**3/81)
delta0 = 0.75*J2*(3*np.cos(np.deg2rad(i))**2 - 1) / (a2**2*beta**3)
n_mean = n0 / (1 + delta0)
a = (ke / n_mean)**(2/3)*Re  # recovered semi-major axis

# Newton-Raphson iteration to find Eccentric Anomaly (E)
E = M.copy()
for _ in range(10):
    E -= (E-e*np.sin(E) - M)/(1 - e*np.cos(E))
theta = np.degrees(np.arctan2(np.sqrt(1-e**2)*np.sin(E), np.cos(E)-e))%360  # true anomaly

# Save mean elements:
mean_elements = pd.DataFrame({
    "epoch_utc": df["EPOCH"],
    "a_mean_km": a,
    "e_mean": e,
    "i_mean_deg": i,
    "raan_mean_deg": raan,
    "omega_mean_deg": omega,
    "M_mean_deg": np.degrees(M),
    "theta_mean_orbit_deg": theta
})

output_path = data_path / "mean-elements"
output_path.mkdir(parents=True, exist_ok=True)
mean_elements.to_csv(output_path / "lageos1_mean_2025.csv", index=False)

print("Number of TLEs:", len(mean_elements))
mean_elements.head()


Number of TLEs: 566


,epoch_utc,a_mean_km,e_mean,i_mean_deg,raan_mean_deg,omega_mean_deg,M_mean_deg,theta_mean_orbit_deg
0,2025-01-01 03:21:13.726080,12270.015281,0.004414,109.8523,357.0926,55.8219,315.9656,315.612637
1,2025-01-02 10:58:27.570720,12270.014804,0.004410,109.8531,357.5433,55.5246,105.2193,105.706215
3,2025-01-07 05:35:44.462400,12270.014859,0.004418,109.8536,359.1775,54.4700,285.9521,285.464563
4,2025-01-08 13:24:55.893600,12270.015006,0.004415,109.8539,359.6312,54.1770,94.2950,94.799297
5,2025-01-09 06:26:36.124224,12270.014895,0.004413,109.8542,359.8740,54.0129,285.5709,285.083061


### Deriving Osculating Elements from $(\vec{r}_{\text{TLE}}, \vec{v}_{\text{TLE}})$ 

Loading the pre-derived positional and velocity vectors $\vec{r}_{\text{TLE}}$ and $\vec{v}_{\text{TLE}}$ calculated and exported into CSV files in a [previous noteboook](01_tle_sp3_residuals.ipynb), we can now plug these into equations for deriving osculating orbital elements, where:

$$
\vec{r}_{\mathrm{TLE}} =
\begin{bmatrix}x\\y\\z\end{bmatrix},
\qquad
\vec{v}_{\mathrm{TLE}} =
\begin{bmatrix}v_x\\v_y\\v_z\end{bmatrix}.
$$


The classical osculating orbital elements are recovered from these vectors using the two-body relations, where all angular elements are in degrees. Using the vis-viva equation, the osculating semi-major axis can be obtained via

$$
a_{\mathrm{osc}} =
\frac{1}{
\frac{2}{r}-\frac{v^2}{\mu}
} \; \; \; \; \; (1)
$$

The eccentricity vector is

$$
\vec{e} =
\frac{\vec{v}_{\mathrm{TLE}}\times\vec{h}}{\mu}
-\frac{\vec{r}_{\mathrm{TLE}}}{r}
$$

Thus, taking the magnitude yields the osculating eccentricity such that

$$
e_{\mathrm{osc}}=\|\vec{e}\| \; \; \; \; \; (2)
$$

The osculating inclination can be found via

$$
i_{\mathrm{osc}} =
\cos^{-1}\left(\frac{h_z}{h}\right) \; \; \; \; \; (3)
$$

The right ascension of the ascending node (RAAN) is obtained through

$$
\Omega_{\mathrm{osc}} =
\begin{cases}
\cos^{-1}\left(\frac{N_x}{N}\right), & N_y\geq0,\; \; \; \; \; (4)\\
2\pi-\cos^{-1}\left(\frac{N_x}{N}\right), & N_y<0.
\end{cases}
$$

The argument of periapsis can be calculated such that
$$
\omega_{\mathrm{osc}} =
\begin{cases}
\cos^{-1}\left(\frac{\mathbf N\cdot\mathbf e}{Ne}\right), & e_z\geq0, \; \; \; \; \; (5)\\
2\pi-\cos^{-1}\left(\frac{\mathbf N\cdot\mathbf e}{Ne}\right),
& e_z<0.
\end{cases}
$$

Finally, the true anomaly of a satellite is obtained using
$$
\theta_{\mathrm{osc}} =
\begin{cases}
\cos^{-1}\left(
\frac{\mathbf e\cdot\mathbf r_{\mathrm{TLE}}}{er}
\right), & \mathbf r_{\mathrm{TLE}}\cdot\mathbf v_{\mathrm{TLE}}\geq0, \; \; \; \; \; (6)\\
2\pi-\cos^{-1}\left(
\frac{\mathbf e\cdot\mathbf r_{\mathrm{TLE}}}{er}
\right), & \mathbf r_{\mathrm{TLE}}\cdot\mathbf v_{\mathrm{TLE}}<0.
\end{cases}
$$

In [5]:
# Load precalculated SGP4 LAGEOS-1 states:
tle_states = pd.read_csv(data_path/"states"/"lageos1_tle_2025.csv", parse_dates=["epoch_utc"])
print("Number of propagated states:", len(tle_states))
tle_states.head()

Number of propagated states: 8328


,epoch_utc,x_m,y_m,z_m,vx_m_s,vy_m_s,vz_m_s
0,2025-01-01 00:00:00,7.603377e+06,-3.635564e+06,8.838050e+06,-4473.341543,-981.651450,3437.456434
1,2025-01-01 01:00:00,-1.037315e+07,-1.687393e+06,6.330673e+06,-3052.490671,1790.807488,-4468.991867
2,2025-01-01 02:00:00,-5.578209e+06,4.020310e+06,-1.023639e+07,5048.709146,614.841017,-2507.569970
3,2025-01-01 03:00:00,1.137938e+07,9.626999e+05,-4.470688e+06,2079.231572,-1909.200237,4955.658083
4,2025-01-01 04:00:00,3.198323e+06,-4.166676e+06,1.102555e+07,-5515.251434,-233.563153,1518.785178


In [6]:
# Get instataneous r and v + useful vectors:
r = np.sqrt(tle_states['x_m']**2 + tle_states['y_m']**2 + tle_states['z_m']**2).to_numpy()
v = np.sqrt(tle_states['vx_m_s']**2 + tle_states['vy_m_s']**2 + tle_states['vz_m_s']**2).to_numpy()
r_vec = tle_states[['x_m', 'y_m', 'z_m']].to_numpy()
v_vec = tle_states[['vx_m_s', 'vy_m_s', 'vz_m_s']].to_numpy()

mu = (6.6743*10**-11)*(5.972*10**24)  # mu = GM
h = np.cross(r_vec, v_vec)  # momentum vector
h_mag = np.linalg.norm(h, axis=1)
z_hat = np.array([0, 0, 1])
N = np.cross(z_hat, h)  # nodal vector (direction of AN)
N_mag = np.linalg.norm(N, axis=1)

# Semi-major axis:
a = 1/((2/r)-(v**2/mu))  # using (1)

# Eccentricity:
e_vec = np.cross(v_vec, h)/mu - r_vec/r[:, None]
e = np.linalg.norm(e_vec, axis=1)  # using (2)

# Inclination:
i = np.degrees(np.arccos(np.clip(h[:, 2]/h_mag, -1, 1)))  # using (3)

# RAAN:
raan = np.degrees(np.arctan2(N[:, 1], N[:, 0]))%360  # using (4)

# Argument of periapsis:
omega = np.degrees(np.arccos(np.clip(np.sum(N*e_vec, axis=1) / (N_mag*e), -1, 1)))
omega = np.where(e_vec[:, 2] < 0, 360 - omega, omega)  # using (5)

# True anomaly:
theta = np.degrees(np.arccos(np.clip(np.sum(e_vec*r_vec, axis=1)/(e*r), -1, 1)))
theta = np.where(np.sum(r_vec*v_vec, axis=1) < 0, 360 - theta, theta)  # using (6)

# Saving osculating orbital elements to CSV:
osc_elements = pd.DataFrame({
    "epoch_utc": tle_states["epoch_utc"],
    "a_osc_km": a*0.001,
    "e_osc": e,
    "i_osc_deg": i,
    "raan_osc_deg": raan,
    "omega_osc_deg": omega,
    "theta_osc_deg": theta
})

osc_path = data_path / "osc-elements"
osc_path.mkdir(parents=True, exist_ok=True)
osc_elements.to_csv(osc_path / "lageos1_osc_2025.csv", index=False)

print("Number of osculating element sets:", len(osc_elements))


Number of osculating element sets: 8328


### Plotting Mean and Osculating Orbit Elements over Time

In [ ]:
# Running script from Jupyter notebook + bypassing kernel crash:
conda_path = Path(sys.prefix)

# Include Conda's DLL directories
env = os.environ.copy()
env["PATH"] = os.pathsep.join([
    str(conda_path),
    str(conda_path / "Library" / "bin"),
    str(conda_path / "Library" / "usr" / "bin"),
    str(conda_path / "Scripts"),
    env.get("PATH", "")
])

# Run plotting script in a separate process
subprocess.run([sys.executable, "plot_osc_mean_elements.py"], env=env, check=True)

CompletedProcess(args=['c:\\Users\\lette\\anaconda3\\envs\\brent\\python.exe', 'plot_osc_mean_elements.py'], returncode=0)

### Bibliography

Hoots, F. R., Schumacher Jr., P. W., Glover, R. A. (2004). _History of Analytical Orbit Modeling in the U.S. Space Surveillance System_. Journal of Guidance, Control, and Dynamics, Vol. 27, No. 2.